# GraphSAGE — Edge Classification Benchmark
## UNSW IoT Botnet · 5-Class Edge-Level Classification


## Section 0 — Configuration

In [1]:
# -- CHANGE THIS -------------------------------------------------------------
GRAPH_PT_PATH = "/content/drive/MyDrive/botnet_graph.pt"
# ----------------------------------------------------------------------------

SEED = 42

# -- GraphSAGE Hyperparameters --------------------------------------------
HP = dict(
    hidden_dim          = 64,
    num_layers          = 3,
    dropout             = 0.3,
    lr                  = 5e-4,
    weight_decay        = 1e-4,
    batch_size          = 2048,
    fan_out             = [10, 10, 10],
    max_epochs          = 100,
    patience            = 15,
    min_delta           = 5e-5,
    val_ratio           = 0.10,
    label_smoothing     = 0.03,
)
print('Hyperparameters:', HP)


Hyperparameters: {'hidden_dim': 64, 'num_layers': 3, 'dropout': 0.3, 'lr': 0.0005, 'weight_decay': 0.0001, 'batch_size': 2048, 'fan_out': [10, 10, 10], 'max_epochs': 100, 'patience': 15, 'min_delta': 5e-05, 'val_ratio': 0.1, 'label_smoothing': 0.03}


## Section 1 — Install Dependencies

In [2]:

import sys, subprocess, re

def pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "-q", *args])

# Remove any previously installed PyG packages that can conflict.
subprocess.call([
    sys.executable, "-m", "pip", "uninstall", "-y",
    "torch-geometric", "pyg-lib", "torch-scatter",
    "torch-sparse", "torch-cluster", "torch-spline-conv"
])

import torch

torch_ver = re.match(r"(\d+\.\d+\.\d+)", torch.__version__).group(1)
cuda_ver = torch.version.cuda

if cuda_ver is None:
    wheel_url = f"https://data.pyg.org/whl/torch-{torch_ver}+cpu.html"
else:
    wheel_url = f"https://data.pyg.org/whl/torch-{torch_ver}+cu{cuda_ver.replace('.', '')}.html"

pip_install(
    "torch-geometric",
    "pyg-lib",
    "torch-scatter",
    "torch-sparse",
    "torch-cluster",
    "torch-spline-conv",
    "-f", wheel_url
)

print("Installed PyG dependencies from:", wheel_url)


Installed PyG dependencies from: https://data.pyg.org/whl/torch-2.11.0+cu128.html


## Section 2 — Imports & Reproducibility

In [3]:

import os, time, warnings
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch_geometric.loader import LinkNeighborLoader
from torch_geometric.nn import SAGEConv

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
warnings.filterwarnings('ignore')

def set_seed(seed: int = 42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")


Device: cuda


## Section 3 — Load Graph

In [4]:

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    pass

data = torch.load(GRAPH_PT_PATH, map_location='cpu', weights_only=False)
print(data)

print(f"\nNodes            : {data.num_nodes}")
print(f"Edges (total)    : {data.edge_index.shape[1]:,}")
print(f"Node feat dim    : {data.x.shape[1]}")
print(f"Edge feat dim    : {data.edge_attr.shape[1]}")
print(f"Num classes      : {data.num_classes}")

NUM_CLASSES   = int(data.num_classes)
NODE_FEAT_DIM = int(data.x.shape[1])
EDGE_FEAT_DIM = int(data.edge_attr.shape[1])

# Expected label order for this benchmark (alphabetical — matches LabelEncoder output).
LABEL_MAP = {0: 'DDoS', 1: 'DoS', 2: 'Normal', 3: 'Reconnaissance', 4: 'Theft'}

# Build a fallback lookup from (src, dst) -> original edge row index.
# This is used only if the loader does not expose input_id.
edge_pair_to_id = {}
for eid, (s, d) in enumerate(data.edge_index.t().tolist()):
    edge_pair_to_id.setdefault((int(s), int(d)), int(eid))
print(f"Built edge pair lookup for {len(edge_pair_to_id):,} unique directed pairs.")


Mounted at /content/drive
Data(x=[93, 6], edge_index=[2, 3668615], edge_attr=[3668615, 15], edge_label=[3668615], train_mask=[3668615], test_mask=[3668615], num_nodes=93, num_classes=5)

Nodes            : 93
Edges (total)    : 3,668,615
Node feat dim    : 6
Edge feat dim    : 15
Num classes      : 5
Built edge pair lookup for 249 unique directed pairs.


## Section 4 — Train / Validation / Test Split

In [5]:

all_train_edge_idx = data.train_mask.nonzero(as_tuple=False).view(-1).long()
test_edge_idx      = data.test_mask.nonzero(as_tuple=False).view(-1).long()

if hasattr(data, 'val_mask') and data.val_mask is not None and int(data.val_mask.sum()) > 0:
    val_edge_idx   = data.val_mask.nonzero(as_tuple=False).view(-1).long()
    train_edge_idx = all_train_edge_idx
    print("Using existing validation mask.")
else:
    # Stratified split from the original training edges only.
    train_labels_np = data.edge_label[all_train_edge_idx].cpu().numpy()
    rel_indices = np.arange(len(all_train_edge_idx))
    tr_rel, val_rel = train_test_split(
        rel_indices,
        test_size=HP['val_ratio'],
        random_state=SEED,
        stratify=train_labels_np,
    )
    train_edge_idx = all_train_edge_idx[tr_rel]
    val_edge_idx   = all_train_edge_idx[val_rel]
    print(f"Created stratified validation split with val_ratio={HP['val_ratio']:.2f}.")

def describe_split(name, edge_idx):
    labels = data.edge_label[edge_idx]
    counts = torch.bincount(labels, minlength=NUM_CLASSES)
    total = int(edge_idx.numel())
    txt = ", ".join([f"{LABEL_MAP[i]}={int(counts[i])}" for i in range(NUM_CLASSES)])
    print(f"{name:<6} edges: {total:>10,} | {txt}")

describe_split("Train", train_edge_idx)
describe_split("Val",   val_edge_idx)
describe_split("Test",  test_edge_idx)


Created stratified validation split with val_ratio=0.10.
Train  edges:  2,641,335 | DDoS=1387183, DoS=1188133, Normal=333, Reconnaissance=65627, Theft=59
Val    edges:    293,482 | DDoS=154132, DoS=132015, Normal=37, Reconnaissance=7292, Theft=6
Test   edges:    733,705 | DDoS=385309, DoS=330112, Normal=107, Reconnaissance=18163, Theft=14


## Section 5 — Loss Function

In [6]:
train_labels = data.edge_label[train_edge_idx].long()
counts = torch.bincount(train_labels, minlength=NUM_CLASSES).float().clamp(min=1)

print("Class counts (train):")
for i in range(NUM_CLASSES):
    print(f"  [{i}] {LABEL_MAP[i]:<20} count={int(counts[i]):>10,}")

# Plain cross-entropy — no class weights.
# Class imbalance is handled by GraphSMOTE oversampling (Section 5b).
criterion = nn.CrossEntropyLoss(label_smoothing=HP['label_smoothing'])
print(f"\nLoss: CrossEntropyLoss(label_smoothing={HP['label_smoothing']})")
print("Class imbalance handled via GraphSMOTE oversampling on the training set (see Section 5b).")


Class counts (train):
  [0] DDoS                 count= 1,387,183
  [1] DoS                  count= 1,188,133
  [2] Normal               count=       333
  [3] Reconnaissance       count=    65,627
  [4] Theft                count=        59

Loss: CrossEntropyLoss(label_smoothing=0.03)
Class imbalance handled via GraphSMOTE oversampling on the training set (see Section 5b).


## Section 5b — GraphSMOTE Oversampling (Training Edges Only)

GraphSMOTE (Zhao et al., 2021) addresses class imbalance by generating synthetic
minority-class embeddings in **feature space** rather than duplicating raw samples.
Applied exclusively to the **training split** — no leakage into validation or test sets.

**Algorithm (edge-level adaptation)**:
1. Build a per-edge feature vector: `concat(proj(x[src]), proj(x[dst]), edge_attr)`,  
   where `proj` is a frozen `Linear(NODE_FEAT_DIM → hidden_dim)` so the vector matches  
   the input shape of `model.edge_mlp` exactly.
2. For each minority class, fit a *k*-NN (k=5) over training edges of that class.
3. Interpolate between each sample and a random neighbour:  
   `x_syn = x_i + λ·(x_j − x_i)`, λ ~ Uniform(0, 1).
4. Upsample until every class matches the majority-class count.
5. Synthetic vectors are consumed by a dedicated DataLoader inside `train_one_epoch`,  
   bypassing the GNN encoder and feeding directly into `model.edge_mlp`.

In [7]:
# ── GraphSMOTE: edge-level oversampling on training split only ───────────────
# Reference: Zhao et al., "GraphSMOTE: Imbalanced Node Classification on Graphs
# with Graph Neural Networks", WSDM 2021.
#
# UNSW IoT Botnet class distribution (train, approx.):
#   DDoS          ~1,387,183   ← majority (no augmentation needed)
#   DoS           ~1,188,133
#   Normal               333   ← extreme minority
#   Reconnaissance    65,627
#   Theft                 59   ← extreme minority
#
# Memory-safe design:
#   • smote_feats stores ONLY new synthetic rows — NOT the original edges.
#     Original edges are already handled by train_loader; storing them again
#     would double RAM consumption for no benefit.
#   • Each minority class is upsampled to at most SMOTE_TARGET_PER_CLASS
#     (absolute ceiling, not a ratio). This keeps smote_feats tiny regardless
#     of majority size.
#   • k-NN uses chunked row-wise distance (no [n×n] matrix) so even the
#     65k Reconnaissance class doesn't blow memory.

import torch
from torch import Tensor
from torch.utils.data import TensorDataset, DataLoader as TorchDataLoader

# ── Tuneable knobs ────────────────────────────────────────────────────────────
SMOTE_TARGET_PER_CLASS = 50_000   # synthetic ceiling per minority class
SMOTE_K                = 5        # nearest neighbours for interpolation
SMOTE_CHUNK            = 256      # rows per chunk in distance computation

# With SMOTE_TARGET_PER_CLASS = 10_000:
#   Normal (333)        → +9,667 synthetic rows
#   Reconnaissance (65k)→ already > 10k, no augmentation
#   Theft (59)          → +9,941 synthetic rows
#   smote_feats total   ≈ 19,608 rows × 136 dims × 4 bytes ≈ 10 MB  ✓

# ── Projection layer: raw node feat → hidden_dim ─────────────────────────────
_smote_hidden   = HP['hidden_dim']
smote_node_proj = nn.Linear(NODE_FEAT_DIM, _smote_hidden, bias=False).to('cpu')
nn.init.xavier_uniform_(smote_node_proj.weight)
smote_node_proj.eval()   # frozen; used only to build smote_feats


def _chunked_knn(cls_feats: Tensor, k: int, chunk: int) -> Tensor:
    """Return nn_idx [n, k] using chunked distances — O(chunk × n × D) peak RAM."""
    n    = cls_feats.shape[0]
    k    = min(k, n - 1)
    nidx = torch.empty(n, k, dtype=torch.long)
    c2   = (cls_feats ** 2).sum(1, keepdim=True).t()   # [1, n]  — precomputed
    for start in range(0, n, chunk):
        end  = min(start + chunk, n)
        rows = cls_feats[start:end]                    # [b, D]
        dot  = rows @ cls_feats.t()                    # [b, n]
        r2   = (rows ** 2).sum(1, keepdim=True)        # [b, 1]
        dist = r2 + c2 - 2 * dot                       # [b, n]
        for li, gi in enumerate(range(start, end)):
            dist[li, gi] = float('inf')                # exclude self
        nidx[start:end] = dist.topk(k, largest=False, dim=1).indices
    return nidx   # [n, k]


def graphsmote_edges(
    x: Tensor,
    edge_index: Tensor,
    edge_attr: Tensor,
    edge_idx: Tensor,
    edge_y: Tensor,
    proj: nn.Module,
    target_per_class: int = SMOTE_TARGET_PER_CLASS,
    k: int             = SMOTE_K,
    chunk: int         = SMOTE_CHUNK,
    seed: int          = 42,
) -> tuple:
    """Return (synthetic_edge_features, synthetic_labels) — originals NOT included.

    synthetic_edge_features : [E_synth, hidden_dim*2 + EDGE_FEAT_DIM]
    synthetic_labels        : [E_synth]
    Only minority classes with count < target_per_class are augmented.
    """
    rng = torch.Generator(); rng.manual_seed(seed)

    src = edge_index[0, edge_idx]
    dst = edge_index[1, edge_idx]
    ea  = edge_attr[edge_idx]
    lab = edge_y[edge_idx].long()

    with torch.no_grad():
        h_src = proj(x[src])
        h_dst = proj(x[dst])
    feats = torch.cat([h_src, h_dst, ea], dim=1)   # [E_train, D]

    num_classes = int(lab.max().item()) + 1
    counts      = torch.bincount(lab, minlength=num_classes)

    synth_feats, synth_labs = [], []

    for cls in range(num_classes):
        n_cls  = int(counts[cls].item())
        n_need = target_per_class - n_cls
        if n_need <= 0:
            print(f"    [{cls}] {LABEL_MAP[cls]:<20} {n_cls:>10,}  (no augmentation needed)")
            continue

        mask      = (lab == cls).nonzero(as_tuple=False).view(-1)
        cls_feats = feats[mask]
        k_eff     = min(k, n_cls - 1) if n_cls > 1 else 1

        if k_eff == 0:
            noise = torch.randn(n_need, cls_feats.shape[1], generator=rng) * 1e-6
            synth = cls_feats.expand(n_need, -1).clone() + noise
        else:
            nn_idx      = _chunked_knn(cls_feats, k_eff, chunk)
            anchor_ids  = torch.randint(0, n_cls,  (n_need,), generator=rng)
            slot_ids    = torch.randint(0, k_eff,  (n_need,), generator=rng)
            lam         = torch.rand(n_need, 1,               generator=rng)
            anchors     = cls_feats[anchor_ids]
            neighbours  = cls_feats[nn_idx[anchor_ids, slot_ids]]
            synth       = anchors + lam * (neighbours - anchors)

        synth_feats.append(synth)
        synth_labs.append(torch.full((n_need,), cls, dtype=torch.long))
        print(f"    [{cls}] {LABEL_MAP[cls]:<20} {n_cls:>10,}  →  {n_cls + n_need:>10,}  (+{n_need:,} synthetic)")

    if not synth_feats:
        # No minority class found — return empty tensors
        d = feats.shape[1]
        return torch.empty(0, d), torch.empty(0, dtype=torch.long)

    synth_feats = torch.cat(synth_feats, dim=0)
    synth_labs  = torch.cat(synth_labs,  dim=0)
    perm        = torch.randperm(synth_feats.shape[0], generator=rng)
    return synth_feats[perm], synth_labs[perm]


print("Running GraphSMOTE on training edges …")
print(f"  Target ceiling : {SMOTE_TARGET_PER_CLASS:,} samples per class")
print(f"  Classes already above ceiling are left untouched.")
print()
print("  Per-class result:")
smote_feats, smote_labels = graphsmote_edges(
    x                = data.x,
    edge_index        = data.edge_index,
    edge_attr         = data.edge_attr,
    edge_idx          = train_edge_idx,
    edge_y            = data.edge_label,
    proj              = smote_node_proj,
    target_per_class  = SMOTE_TARGET_PER_CLASS,
    k                 = SMOTE_K,
    chunk             = SMOTE_CHUNK,
    seed              = SEED,
)
expected_dim = _smote_hidden * 2 + EDGE_FEAT_DIM
assert smote_feats.shape[1] == expected_dim, \
    f"Dim mismatch: smote_feats={smote_feats.shape[1]}, edge_mlp expects={expected_dim}"
print()
print(f"  Synthetic rows only : {smote_feats.shape[0]:>10,}")
print(f"  Feature dim         : {smote_feats.shape[1]}  (= hidden_dim*2 + EDGE_FEAT_DIM) ✓")
mem_mb = smote_feats.numel() * 4 / 1e6
print(f"  smote_feats RAM     : {mem_mb:.1f} MB")

smote_dataset = TensorDataset(smote_feats, smote_labels)
smote_loader  = TorchDataLoader(
    smote_dataset,
    batch_size=HP['batch_size'],
    shuffle=True,
    drop_last=False,
)
print(f"  SMOTE DataLoader    : {len(smote_loader)} batches  ✓")


Running GraphSMOTE on training edges …
  Target ceiling : 50,000 samples per class
  Classes already above ceiling are left untouched.

  Per-class result:
    [0] DDoS                  1,387,183  (no augmentation needed)
    [1] DoS                   1,188,133  (no augmentation needed)
    [2] Normal                      333  →      50,000  (+49,667 synthetic)
    [3] Reconnaissance           65,627  (no augmentation needed)
    [4] Theft                        59  →      50,000  (+49,941 synthetic)

  Synthetic rows only :     99,608
  Feature dim         : 143  (= hidden_dim*2 + EDGE_FEAT_DIM) ✓
  smote_feats RAM     : 57.0 MB
  SMOTE DataLoader    : 49 batches  ✓


## Section 6 — DataLoaders

> **Fix applied (v2)**: `make_loader` now accepts `mp_edge_idx` to restrict
> the message-passing graph to training edges only during train and val.
> This eliminates structural leakage where test-set edges could appear in
> neighborhood sampling context while computing node embeddings.
> The test loader still uses the full graph (training complete, no gradients flow).

In [8]:
def make_loader(edge_idx, batch_size, shuffle, mp_edge_idx=None):
    """
    Build a LinkNeighborLoader.

    mp_edge_idx : indices of edges used for message-passing (MP graph).
                  Pass train_edge_idx for train/val loaders so test edges
                  never appear in the MP neighborhood during training
                  (eliminates structural leakage).
                  Pass None for the test loader — training is complete,
                  no gradients flow, so full-graph sampling is safe.
    """
    edge_idx = edge_idx.long()
    if mp_edge_idx is not None:
        mp_edge_index = data.edge_index[:, mp_edge_idx.long()]
    else:
        mp_edge_index = data.edge_index

    from torch_geometric.data import Data as PygData
    mp_data = PygData(
        x=data.x,
        edge_index=mp_edge_index,
        edge_attr=data.edge_attr,
        num_nodes=data.num_nodes,
    )

    return LinkNeighborLoader(
        mp_data,
        num_neighbors=HP['fan_out'],
        edge_label_index=data.edge_index[:, edge_idx],
        edge_label=data.edge_label[edge_idx].long(),
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
    )

# Train loader  : MP graph contains only training edges.
# Val loader    : val edges scored but not in the MP graph.
# Test loader   : training done, full graph safe (no gradient flow).


def resolve_labeled_edge_attr(batch, seed_edge_idx, full_edge_attr, pair_lookup):
    """
    Fetch edge attributes for the labeled (seed) edges in this batch.

    Preferred: batch.input_id -> seed_edge_idx -> global edge row.
    Fallback : (src, dst) pair lookup (used if input_id is absent).
    """
    if hasattr(batch, "input_id") and batch.input_id is not None:
        input_id = batch.input_id.detach().cpu().long()
        seed_edge_idx_cpu = seed_edge_idx.detach().cpu().long()
        global_edge_ids = seed_edge_idx_cpu[input_id]
        return full_edge_attr[global_edge_ids].to(batch.x.device)

    edge_label_index = batch.edge_label_index
    if hasattr(batch, "n_id"):
        if int(edge_label_index.max()) < int(batch.n_id.numel()):
            src = batch.n_id[edge_label_index[0].detach().cpu().long()]
            dst = batch.n_id[edge_label_index[1].detach().cpu().long()]
        else:
            src = edge_label_index[0].detach().cpu().long()
            dst = edge_label_index[1].detach().cpu().long()
    else:
        src = edge_label_index[0].detach().cpu().long()
        dst = edge_label_index[1].detach().cpu().long()

    edge_ids = []
    for s, d in zip(src.tolist(), dst.tolist()):
        eid = pair_lookup.get((int(s), int(d)))
        if eid is None:
            raise KeyError(f"Could not resolve edge attribute for pair ({s}, {d}).")
        edge_ids.append(eid)

    edge_ids = torch.tensor(edge_ids, dtype=torch.long)
    return full_edge_attr[edge_ids].to(batch.x.device)

print("Edge-attribute resolver ready.")


Edge-attribute resolver ready.


## Section 7 — GraphSAGE Model Definition

In [9]:
class GraphSAGEEdgeClassifier(nn.Module):
    """
    GraphSAGE edge-level classifier.

    Node encoder   : Linear projection
    SAGEConv stack  : graph convolution
    Edge scorer    : MLP([h_src || h_dst || edge_attr]) -> class logits
    """
    def __init__(self, node_feat_dim, edge_feat_dim, hidden_dim,
                 num_layers, num_classes, dropout):
        super().__init__()
        self.dropout = dropout

        self.node_encoder = nn.Linear(node_feat_dim, hidden_dim)

        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()

        for _ in range(num_layers):
            self.convs.append(
                SAGEConv(hidden_dim, hidden_dim)
            )
            self.norms.append(nn.LayerNorm(hidden_dim))

        mlp_in = hidden_dim * 2 + edge_feat_dim

        self.edge_mlp = nn.Sequential(
            nn.Linear(mlp_in, hidden_dim * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def encode_nodes(self, x, edge_index):
        h = F.relu(self.node_encoder(x))
        for conv, norm in zip(self.convs, self.norms):
            h = conv(h, edge_index)
            h = norm(h)
            h = F.relu(h)
            h = F.dropout(h, p=self.dropout, training=self.training)
        return h

    def forward(self, x, edge_index, edge_attr, edge_label_index):
        h = self.encode_nodes(x, edge_index)
        src, dst = edge_label_index
        edge_repr = torch.cat([h[src], h[dst], edge_attr], dim=-1)
        return self.edge_mlp(edge_repr)




## Section 9 — Training & Validation Loop

> **Fix applied (v2)**: Early stopping now uses **val_f1 as the single criterion**.
> The previous dual-criterion logic reset patience when either val_loss or val_f1
> improved, keeping training alive on negligible deltas past the true optimum.
> Single-criterion stopping is cleaner and consistent with standard practice.

In [10]:
def train_one_epoch(model, loader, optimizer, criterion, device, seed_edge_idx):
    model.train()
    total_loss, total_correct, total_edges = 0.0, 0, 0

    # ── (1) Real graph mini-batches: full GNN pipeline ───────────────────────
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()

        edge_attr = resolve_labeled_edge_attr(
            batch=batch,
            seed_edge_idx=seed_edge_idx,
            full_edge_attr=data.edge_attr,
            pair_lookup=edge_pair_to_id,
        )

        out = model(
            batch.x,
            batch.edge_index,
            edge_attr,
            batch.edge_label_index,
        )
        labels = batch.edge_label.long()
        loss = criterion(out, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        preds = out.argmax(dim=-1)
        total_correct += (preds == labels).sum().item()
        total_loss    += loss.item() * labels.size(0)
        total_edges   += labels.size(0)

    # ── (2) GraphSMOTE batches: synthetic edges → edge MLP decoder only ──────
    # Synthetic feature vectors = [proj(x_src) || proj(x_dst) || edge_attr].
    # Graph convolution is skipped; they feed directly into model.edge_mlp.
    for sf, sy in smote_loader:
        sf, sy = sf.to(device), sy.to(device)
        optimizer.zero_grad()
        out    = model.edge_mlp(sf)
        loss   = criterion(out, sy)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        preds = out.argmax(dim=-1)
        total_correct += (preds == sy).sum().item()
        total_loss    += loss.item() * sy.size(0)
        total_edges   += sy.size(0)

    return total_loss / total_edges, total_correct / total_edges


@torch.no_grad()
def evaluate(model, loader, criterion, device, seed_edge_idx):
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    total_edges = 0

    for batch in loader:
        batch = batch.to(device)

        edge_attr = resolve_labeled_edge_attr(
            batch=batch,
            seed_edge_idx=seed_edge_idx,
            full_edge_attr=data.edge_attr,
            pair_lookup=edge_pair_to_id,
        )

        out = model(
            batch.x,
            batch.edge_index,
            edge_attr,
            batch.edge_label_index,
        )
        labels = batch.edge_label.long()
        loss   = criterion(out, labels)

        total_loss  += loss.item() * labels.size(0)
        total_edges += labels.size(0)
        all_preds.append(out.argmax(dim=-1).cpu())
        all_labels.append(labels.cpu())

    all_preds  = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    avg_loss = total_loss / total_edges
    acc      = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return avg_loss, acc, macro_f1, all_preds, all_labels


## Section — Multi-Seed Evaluation (added)

In [11]:
# ============================================================
# Section 14 — Multi-Seed Evaluation (GraphSAGE, bot_iot, GraphSMOTE)
# ============================================================
import copy, json, os
import numpy as np
import pandas as pd

MODEL_CLASS  = GraphSAGEEdgeClassifier
MODEL_TAG    = "GraphSAGE"
DATASET_TAG  = "bot_iot"
STRATEGY_TAG = "GraphSMOTE"

SEEDS = [0, 1]

def build_smote(train_edge_idx_r, seed):
    """Re-run GraphSMOTE oversampling for this seed (reuses graphsmote_edges/_chunked_knn
    already defined in Section 5b of this notebook)."""
    proj = nn.Linear(NODE_FEAT_DIM, HP['hidden_dim'], bias=False).to('cpu')
    nn.init.xavier_uniform_(proj.weight)
    proj.eval()
    sf, sy = graphsmote_edges(
        x=data.x, edge_index=data.edge_index, edge_attr=data.edge_attr,
        edge_idx=train_edge_idx_r, edge_y=data.edge_label, proj=proj,
        target_per_class=SMOTE_TARGET_PER_CLASS, k=SMOTE_K, chunk=SMOTE_CHUNK, seed=seed,
    )
    ds = TensorDataset(sf, sy)
    return TorchDataLoader(ds, batch_size=HP['batch_size'], shuffle=True, drop_last=False)

def train_one_epoch_smote(model, loader, smote_loader_r, optimizer, criterion, device, seed_edge_idx):
    model.train()
    total_loss, total_correct, total_edges = 0.0, 0, 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        edge_attr = resolve_labeled_edge_attr(batch=batch, seed_edge_idx=seed_edge_idx,
                                               full_edge_attr=data.edge_attr, pair_lookup=edge_pair_to_id)
        out = model(batch.x, batch.edge_index, edge_attr, batch.edge_label_index)
        labels = batch.edge_label.long()
        loss = criterion(out, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        preds = out.argmax(dim=-1)
        total_correct += (preds == labels).sum().item()
        total_loss    += loss.item() * labels.size(0)
        total_edges   += labels.size(0)

    for sf, sy in smote_loader_r:
        sf, sy = sf.to(device), sy.to(device)
        optimizer.zero_grad()
        out  = model.edge_mlp(sf)
        loss = criterion(out, sy)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        preds = out.argmax(dim=-1)
        total_correct += (preds == sy).sum().item()
        total_loss    += loss.item() * sy.size(0)
        total_edges   += sy.size(0)

    return total_loss / total_edges, total_correct / total_edges

def run_once(seed):
    set_seed(seed)

    if hasattr(data, 'val_mask') and data.val_mask is not None and int(data.val_mask.sum()) > 0:
        val_edge_idx_r   = data.val_mask.nonzero(as_tuple=False).view(-1).long()
        train_edge_idx_r = all_train_edge_idx
    else:
        train_labels_np = data.edge_label[all_train_edge_idx].cpu().numpy()
        rel_indices = np.arange(len(all_train_edge_idx))
        tr_rel, val_rel = train_test_split(
            rel_indices, test_size=HP['val_ratio'],
            random_state=seed, stratify=train_labels_np,
        )
        train_edge_idx_r = all_train_edge_idx[tr_rel]
        val_edge_idx_r   = all_train_edge_idx[val_rel]

    train_labels_r = data.edge_label[train_edge_idx_r].long()
    counts_r = torch.bincount(train_labels_r, minlength=NUM_CLASSES).float().clamp(min=1)
    criterion_r = nn.CrossEntropyLoss(label_smoothing=HP['label_smoothing'])

    smote_loader_r = build_smote(train_edge_idx_r, seed)

    train_loader_r = make_loader(train_edge_idx_r, HP['batch_size'],     shuffle=True,  mp_edge_idx=train_edge_idx_r)
    val_loader_r   = make_loader(val_edge_idx_r,   HP['batch_size'] * 2, shuffle=False, mp_edge_idx=train_edge_idx_r)
    test_loader_r  = make_loader(test_edge_idx,    HP['batch_size'] * 2, shuffle=False, mp_edge_idx=None)

    model_r = MODEL_CLASS(
        node_feat_dim = NODE_FEAT_DIM,
        edge_feat_dim = EDGE_FEAT_DIM,
        hidden_dim    = HP['hidden_dim'],
        num_layers    = HP['num_layers'],
        num_classes   = NUM_CLASSES,
        dropout       = HP['dropout'],
    ).to(DEVICE)

    optimizer_r = torch.optim.Adam(model_r.parameters(), lr=HP['lr'], weight_decay=HP['weight_decay'])
    scheduler_r = ReduceLROnPlateau(optimizer_r, mode='min', factor=0.5, patience=5, min_lr=1e-6)

    best_val_f1, patience_count, best_state = -float('inf'), 0, None
    history = []
    for epoch in range(1, HP['max_epochs'] + 1):
        set_seed(seed + epoch)
        train_loss, train_acc = train_one_epoch_smote(model_r, train_loader_r, smote_loader_r, optimizer_r, criterion_r, DEVICE, train_edge_idx_r)
        val_loss, val_acc, val_f1, _, _ = evaluate(model_r, val_loader_r, criterion_r, DEVICE, val_edge_idx_r)
        scheduler_r.step(val_loss)
        history.append(dict(epoch=epoch, train_loss=train_loss, train_acc=train_acc,
                             val_loss=val_loss, val_acc=val_acc, val_f1=val_f1))

        if val_f1 > best_val_f1 + HP['min_delta']:
            best_val_f1 = val_f1
            patience_count = 0
            best_state = {k: v.detach().cpu().clone() for k, v in model_r.state_dict().items()}
        else:
            patience_count += 1
        if patience_count >= HP['patience']:
            break

    model_r.load_state_dict(best_state)
    _, test_acc, test_macro_f1, y_pred, y_true = evaluate(model_r, test_loader_r, criterion_r, DEVICE, test_edge_idx)

    test_precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
    test_recall    = recall_score(y_true, y_pred, average='macro', zero_division=0)

    per_class_f1        = f1_score(y_true, y_pred, average=None, zero_division=0, labels=list(range(NUM_CLASSES)))
    per_class_precision = precision_score(y_true, y_pred, average=None, zero_division=0, labels=list(range(NUM_CLASSES)))
    per_class_recall    = recall_score(y_true, y_pred, average=None, zero_division=0, labels=list(range(NUM_CLASSES)))

    per_class = {
        LABEL_MAP[i]: dict(f1=float(per_class_f1[i]), precision=float(per_class_precision[i]), recall=float(per_class_recall[i]))
        for i in range(NUM_CLASSES)
    }

    return dict(
        seed=seed, epochs_ran=epoch,
        acc=float(test_acc), macro_f1=float(test_macro_f1),
        precision=float(test_precision), recall=float(test_recall),
        per_class=per_class,
        history=history,
    )

results = [run_once(s) for s in SEEDS]

training_logs = dict(
    dataset=DATASET_TAG, strategy=STRATEGY_TAG, model=MODEL_TAG,
    seeds=SEEDS,
    per_seed_history={r['seed']: r.pop('history') for r in results},
)

def agg_mean_std(vals):
    arr = np.array(vals, dtype=float)
    return dict(mean=float(arr.mean()), std=float(arr.std(ddof=1)) if len(arr) > 1 else 0.0)

overall = {metric: agg_mean_std([r[metric] for r in results]) for metric in ['acc', 'macro_f1', 'precision', 'recall']}

per_class_agg = {}
for cls in LABEL_MAP.values():
    per_class_agg[cls] = {metric: agg_mean_std([r['per_class'][cls][metric] for r in results]) for metric in ['f1', 'precision', 'recall']}

summary = dict(
    dataset=DATASET_TAG, strategy=STRATEGY_TAG, model=MODEL_TAG,
    seeds=SEEDS, n_seeds=len(SEEDS),
    per_seed_results=results, overall=overall, per_class=per_class_agg,
)

print(f"\n{'='*60}")
print(f"  {MODEL_TAG} — {DATASET_TAG} / {STRATEGY_TAG} — {len(SEEDS)}-seed summary")
print(f"{'='*60}")
for metric, stat in overall.items():
    print(f"  {metric:<10}: {stat['mean']:.4f} ± {stat['std']:.4f}")
print("\n  Per-class F1 (mean ± std):")
for cls, stat in per_class_agg.items():
    f1s = stat['f1']
    print(f"    {cls:<16}: {f1s['mean']:.4f} ± {f1s['std']:.4f}")

out_dir = "/content/results" if os.path.isdir("/content") else "."
os.makedirs(out_dir, exist_ok=True)
out_path = f"{out_dir}/{DATASET_TAG}_{STRATEGY_TAG}_{MODEL_TAG}_multiseed.json"
with open(out_path, "w") as f:
    json.dump(summary, f, indent=2)
print(f"\nSaved: {out_path}")

logs_path = f"{out_dir}/{DATASET_TAG}_{STRATEGY_TAG}_{MODEL_TAG}_trainlogs.json"
with open(logs_path, "w") as f:
    json.dump(training_logs, f, indent=2)
print(f"Saved: {logs_path}")

try:
    from google.colab import files as colab_files
    colab_files.download(out_path)
    colab_files.download(logs_path)
except ImportError:
    pass


    [0] DDoS                  1,387,183  (no augmentation needed)
    [1] DoS                   1,188,133  (no augmentation needed)
    [2] Normal                      333  →      50,000  (+49,667 synthetic)
    [3] Reconnaissance           65,627  (no augmentation needed)
    [4] Theft                        59  →      50,000  (+49,941 synthetic)
    [0] DDoS                  1,387,183  (no augmentation needed)
    [1] DoS                   1,188,133  (no augmentation needed)
    [2] Normal                      333  →      50,000  (+49,667 synthetic)
    [3] Reconnaissance           65,627  (no augmentation needed)
    [4] Theft                        59  →      50,000  (+49,941 synthetic)

  GraphSAGE — bot_iot / GraphSMOTE — 2-seed summary
  acc       : 0.9983 ± 0.0000
  macro_f1  : 0.7745 ± 0.0027
  precision : 0.7724 ± 0.0031
  recall    : 0.7769 ± 0.0090

  Per-class F1 (mean ± std):
    DDoS            : 0.9987 ± 0.0000
    DoS             : 0.9987 ± 0.0000
    Normal          :

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>